# 07 · Topic modeling sul Guardian: testo breve contro testo completo

Il topic modeling del NYT (notebook 04, §5) lavora su un testo breve, titolo, abstract e paragrafo d'apertura, perché il testo integrale del NYT non è accessibile. Il Guardian fornisce invece il testo completo (`body_text`, notebook 05). Questo notebook usa la differenza per rispondere a una domanda di metodo: **i topic estratti dal testo completo sono migliori di quelli estratti dal testo breve?** Il §5 scompone "migliori" in misure separate: stabilità tra esecuzioni, coerenza delle parole, diversità, separazione dei gruppi.

**Confronto controllato.** Confrontare il Guardian a testo completo con il NYT a testo breve non isolerebbe l'effetto della lunghezza, perché cambierebbero insieme giornale, corpus e testo. Il confronto si fa quindi sullo stesso corpus Guardian e con gli stessi parametri BERTopic del NYT, cambiando solo il testo dato al modello:

- **breve**: titolo + sottotitolo + prime `LEAD_WORDS` (60) parole del corpo, costruito per avvicinarsi al testo disponibile per il NYT;
- **completo**: titolo + sottotitolo + tutto il corpo.

**Corpus.** Solo gli articoli `is_ice_relevant` di `articles_ice_relevant.parquet` (filtro di rilevanza del notebook 05, §5). Un topic model sulle citazioni di passaggio troverebbe i temi di altri articoli, non quelli di ICE.

**Il problema del troncamento.** Il modello di embedding `all-mpnet-base-v2` legge al massimo 384 token, circa 280 parole, e ignora il resto senza segnalarlo. Dargli un articolo di 1.000 parole equivale a dargli il suo attacco, cioè di nuovo una versione breve. Per usare davvero il testo completo l'articolo viene diviso in blocchi che stanno sotto il limite (§2); ogni blocco riceve un embedding e l'articolo è rappresentato dalla media dei suoi blocchi (§3).

**Struttura.** §0 setup → §1 corpus → §2 versioni del testo e blocchi → §3 embedding → §4 BERTopic su più seed → §5 confronto breve vs completo → §6 topic del modello di riferimento → §7 salvataggio → §8 lettura dei risultati.

## 0. Setup

Import, percorsi e costanti. Il notebook legge solo dati derivati (`articles_ice_relevant.parquet`) e non chiama nessuna API. Tre scelte fissate qui valgono per tutto il notebook:

- `EMBEDDING_MODEL = "all-mpnet-base-v2"`: lo stesso modello di embedding del topic modeling NYT (notebook 04) e dei commenti (notebook 09), così le differenze tra i modelli non dipendono dal modello di embedding. Fino al 2026-10-13 il notebook usava `all-MiniLM-L6-v2` (come il filtro di rilevanza del notebook 05, che resta su MiniLM); i topic di quella esecuzione sono conservati in `ice_topics_minilm.parquet` e `ice_topic_info_minilm.csv`;
- `PERIOD_ORDER`: i quattro periodi del progetto (notebook 02), usati nelle tabelle e nella heatmap del §6.1;
- `CHUNK_EMBED_CACHE`: cache su disco degli embedding dei blocchi (§3), che su CPU richiedono qualche minuto.

Le librerie sono quelle del topic modeling NYT: `sentence-transformers` per gli embedding, UMAP per ridurre la dimensione, BERTopic (che usa HDBSCAN) per il clustering e le parole rappresentative, `adjusted_mutual_info_score` per misurare la stabilità tra seed.

In [ ]:
import logging
import re
from itertools import combinations
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer, ENGLISH_STOP_WORDS
from sklearn.metrics import adjusted_mutual_info_score
from umap import UMAP

%matplotlib inline

# Radice del progetto: funziona sia lanciando da notebooks/ sia dalla radice
BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROCESSED_DIR = BASE_DIR / "data" / "processed" / "guardian"
PLOTS_DIR = PROCESSED_DIR / "plots"
FIG_DIR = BASE_DIR / "docs" / "figures"  # figure per la tesi
# Input: corpus Guardian con il filtro di rilevanza del notebook 05 (si usano solo gli is_ice_relevant)
IN_RELEVANT = PROCESSED_DIR / "articles_ice_relevant.parquet"
# Cache degli embedding dei blocchi (§3): evita di ricalcolarli a ogni esecuzione
CHUNK_EMBED_CACHE = PROCESSED_DIR / "topic_chunk_embeddings_mpnet.parquet"
# Output (§7): topic per articolo e tabella descrittiva dei topic
OUT_TOPICS = PROCESSED_DIR / "ice_topics.parquet"
OUT_TOPIC_INFO = PROCESSED_DIR / "ice_topic_info.csv"

# Stesso modello di embedding dei notebook 04 e 09: le differenze non dipendono dagli embedding
EMBEDDING_MODEL = "all-mpnet-base-v2"
# I quattro periodi del progetto (notebook 02), nell'ordine delle tabelle
PERIOD_ORDER = ["2003-2016", "2017-2020", "2021-2024", "2025-presente"]

sns.set_theme(style="whitegrid", palette="muted")
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s", force=True)
log = logging.getLogger("guardian_topics")
# Silenzia i log informativi delle librerie, lasciando solo quelli del notebook
for noisy in ("httpx", "sentence_transformers", "huggingface_hub", "BERTopic"):
    logging.getLogger(noisy).setLevel(logging.WARNING)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)
pd.set_option("display.max_colwidth", 140)
print(f"BASE_DIR: {BASE_DIR}")

## 1. Corpus

Il corpus è il sottoinsieme rilevante del notebook 05: i 1.738 articoli, su 3.292 che nominano l'agenzia, che il classificatore di rilevanza (modello B) considera articoli su ICE. Di ogni articolo si usano titolo, sottotitolo (`standfirst`, ripulito dai tag HTML) e corpo.

**Live-blog.** Gli articoli in diretta ("as it happened") sono molto lunghi e raccolgono decine di notizie diverse nella stessa pagina, quindi non hanno un tema unico. Restano nel corpus, per non introdurre una seconda selezione, ma sono marcati (`is_live`, riconosciuti dal percorso `/live/` nell'id) per vedere in quali topic finiscono (§6).

In [ ]:
# Lo standfirst contiene HTML: i tag vanno tolti prima di usarlo come testo
TAG_RE = re.compile(r"<[^>]+>")

df = pd.read_parquet(IN_RELEVANT)
# Solo gli articoli su ICE secondo il filtro di rilevanza (1.738 su 3.292)
df = df[df["is_ice_relevant"]].reset_index(drop=True)
df["standfirst_clean"] = df["standfirst"].fillna("").str.replace(TAG_RE, " ", regex=True).str.strip()
# Live-blog riconosciuti dal percorso dell'id: restano nel corpus, marcati per l'analisi del §6
df["is_live"] = df["id"].str.contains("/live/", regex=False)
# Lunghezza del corpo in parole, per descrivere il corpus
df["n_words"] = df["body_text"].fillna("").str.split().str.len()
log.info("Articoli rilevanti: %d (live-blog: %d)", len(df), df["is_live"].sum())

display(df.groupby("period").agg(articoli=("id", "size"), live=("is_live", "sum"), parole_mediana=("n_words", "median")).reindex(PERIOD_ORDER))
df["n_words"].describe(percentiles=[0.1, 0.5, 0.9]).round(0).to_frame("parole per articolo").T

Il log riporta 1.738 articoli, di cui 39 live-blog (2,2%). Tra i 3.292 articoli che nominano l'agenzia i live-blog erano 409 (12,4%): il filtro di rilevanza ne ha esclusi più del 90%, coerentemente con il fatto che citano ICE tra molte altre notizie.

Prima tabella: articoli, live-blog e lunghezza mediana per periodo. Il 72% degli articoli è del 2025-presente (1.257); i periodi precedenti ne hanno tra 90 e 283. Per questo le quote per periodo del §6.1 sono normalizzate per colonna e vanno lette con cautela. La lunghezza mediana è simile in tutti i periodi (795-899 parole), quindi il confronto tra testo breve e completo non dipende da articoli più lunghi in un periodo.

Seconda tabella: distribuzione della lunghezza. Mediana 862 parole, il 10% degli articoli sotto le 465 e il 10% sopra le 1.760; il massimo, 14.614 parole, è un live-blog. Quasi ogni articolo ha molte più parole delle circa 190 che il modello di embedding legge in una volta.

## 2. Le due versioni del testo e i blocchi

**Le due versioni.**

- `text_short`: titolo + sottotitolo + prime `LEAD_WORDS = 60` parole del corpo. Il sottotitolo del Guardian svolge la funzione dell'abstract del NYT, le prime 60 parole quella del paragrafo d'apertura.
- `text_full`: titolo + sottotitolo + tutto il corpo.

La versione breve si avvicina alla condizione del NYT ma non la replica: in mediana conta 91 parole, contro 39 del testo NYT. Il testo NYT è a sua volta disomogeneo, 74 parole prima del 2025 e 34 dal 2025, perché dal 2025 l'archivio NYT non fornisce più il paragrafo d'apertura (verificato il 2026-10-08). Il confronto misura quindi l'effetto di passare da circa 90 a circa 900 parole sullo stesso corpus, e la differenza di lunghezza rispetto al NYT va ricordata quando i risultati vengono riportati al NYT.

**I blocchi.** Per il testo completo l'articolo viene diviso in blocchi di frasi intere, ciascuno di al più `CHUNK_WORDS = 150` parole. La scelta risponde a tre vincoli:

1. il blocco deve stare sotto il limite di 384 token del modello, altrimenti la parte finale verrebbe troncata. Con 150 parole i blocchi hanno in mediana 172 token e al massimo 235 (controllo del 2026-10-13 con il tokenizer di mpnet): nessun blocco è troncato. La dimensione dei blocchi è quella scelta per MiniLM (limite 256 token) e non è stata allargata al passaggio a mpnet, così il cambio di modello è l'unica differenza rispetto alla prima esecuzione;
2. il blocco deve contenere frasi intere, perché l'embedding di una frase spezzata a metà ne rappresenta male il contenuto;
3. il `body_text` del Guardian non ha a capo, quindi i paragrafi non sono un'unità utilizzabile. Le frasi si individuano con un'espressione regolare (`SENT_SPLIT_RE`) che divide dopo `.`, `!` o `?` seguiti da uno spazio e da una maiuscola o da virgolette.

Le frasi si accumulano finché la successiva non farebbe superare le 150 parole; a quel punto il blocco si chiude e ne inizia uno nuovo. Una frase più lunga di 150 parole forma un blocco da sola. Titolo e sottotitolo fanno parte del primo blocco.

**Limite della divisione in frasi.** L'espressione regolare divide anche dopo un'abbreviazione puntata seguita da una maiuscola, come "U.S. Immigration" (34 occorrenze in tutto il corpus: lo stile del Guardian scrive di norma "US" senza punti). L'effetto è solo uno spostamento del confine tra due blocchi: nessuna parola va persa.

In [ ]:
# Parole del corpo nella versione breve: con titolo e sottotitolo, ~90 parole in mediana
LEAD_WORDS = 60
# Parole massime per blocco: ~172 token in mediana, sotto il limite di 384 del modello
CHUNK_WORDS = 150
# Fine frase: dopo . ! ? (con eventuali virgolette di chiusura), seguiti da spazio e da maiuscola o virgolette
SENT_SPLIT_RE = re.compile(r"(?<=[.!?])[\"”’']?\s+(?=[A-Z“‘\"'])")

# Titolo e sottotitolo, comuni alle due versioni del testo
head = (df["headline"].fillna("") + ". " + df["standfirst_clean"]).str.strip()
body = df["body_text"].fillna("")
# Versione breve: titolo + sottotitolo + prime LEAD_WORDS parole del corpo
df["text_short"] = head + " " + body.map(lambda b: " ".join(b.split()[:LEAD_WORDS]))
# Versione completa: titolo + sottotitolo + tutto il corpo
df["text_full"] = head + " " + body


def chunk_text(text: str, max_words: int = CHUNK_WORDS) -> list[str]:
    """Divide un testo in blocchi di frasi intere, ciascuno di al più max_words parole.

    Scorre le frasi in ordine e le accumula nel blocco corrente; quando la frase successiva
    farebbe superare max_words, chiude il blocco e ne apre uno nuovo. Una frase più lunga di
    max_words forma un blocco da sola. Nessuna parola viene scartata: i blocchi, uniti,
    ricostruiscono il testo (a meno degli spazi).
    """
    chunks, current, n = [], [], 0
    for sent in SENT_SPLIT_RE.split(text):
        w = len(sent.split())
        if current and n + w > max_words:
            chunks.append(" ".join(current))
            current, n = [], 0
        current.append(sent)
        n += w
    if current:
        chunks.append(" ".join(current))
    return chunks


# Una riga per blocco: indice dell'articolo, posizione del blocco nell'articolo, testo
chunks = pd.DataFrame(
    [(i, j, c) for i, text in enumerate(df["text_full"]) for j, c in enumerate(chunk_text(text))],
    columns=["doc_idx", "chunk_idx", "text"],
)
per_doc = chunks.groupby("doc_idx").size()
log.info("Blocchi: %d (mediana %d per articolo, max %d)", len(chunks), per_doc.median(), per_doc.max())
print("parole per blocco:", chunks["text"].str.split().str.len().describe(percentiles=[0.5, 0.95]).round(0).to_dict())
print(f"parole mediane: breve {df['text_short'].str.split().str.len().median():.0f} · completo {df['text_full'].str.split().str.len().median():.0f}")

## 3. Embedding

Ogni articolo viene trasformato in un vettore di 768 dimensioni con `all-mpnet-base-v2`, in due modi:

- **breve**: un embedding per articolo, calcolato su `text_short`. Il testo breve occupa in mediana 115 token e nessun articolo supera il limite di 384: il modello lo legge per intero.
- **completo**: un embedding per ciascuno dei 15.862 blocchi del §2; l'embedding dell'articolo è la media degli embedding dei suoi blocchi, rinormalizzata a lunghezza 1. Ogni blocco pesa allo stesso modo, quindi l'articolo è rappresentato da tutto il suo contenuto e non solo dall'attacco.

Tutti gli embedding sono normalizzati (`normalize_embeddings=True`): il prodotto scalare tra due vettori è la loro similarità del coseno, la misura usata da UMAP (`metric="cosine"`) e dalla riassegnazione degli outlier.

**Effetto della media.** Mediare molti blocchi avvicina gli articoli tra loro: i contenuti specifici dei blocchi si compensano e resta la componente comune, che in un corpus interamente su ICE è ampia. La similarità del coseno media tra due articoli è 0,54 con il testo completo e 0,40 con il breve (controllo del 2026-10-13 con mpnet; con MiniLM era 0,49 e 0,30). La conseguenza si vede nel §4: la soglia di 0,4 usata per riassegnare gli outlier al topic più simile, ripresa dal NYT dove gli embedding sono brevi, con gli embedding completi non filtra quasi nulla.

**Cache.** Gli embedding dei blocchi richiedono qualche minuto su CPU e sono salvati in `topic_chunk_embeddings_mpnet.parquet` insieme a un'impronta (hash) del testo dei blocchi: vengono riusati solo se i blocchi sono esattamente gli stessi, altrimenti si ricalcolano.

In [ ]:
# Modello caricato una volta e riusato per entrambe le versioni e da BERTopic (§4)
embedding_model = SentenceTransformer(EMBEDDING_MODEL)


def embed_chunks() -> np.ndarray:
    """Restituisce gli embedding normalizzati di tutti i blocchi, uno per riga, usando la cache se valida.

    La cache è valida solo se ha lo stesso numero di blocchi e la stessa impronta (hash) del loro
    testo; altrimenti gli embedding vengono ricalcolati con il modello e la cache viene riscritta.
    """
    key = pd.util.hash_pandas_object(chunks["text"], index=False).sum()
    if CHUNK_EMBED_CACHE.exists():
        cached = pd.read_parquet(CHUNK_EMBED_CACHE)
        if len(cached) == len(chunks) and cached.attrs.get("key") == str(key):
            return cached.to_numpy()
    log.info("Embedding di %d blocchi con %s...", len(chunks), EMBEDDING_MODEL)
    vecs = embedding_model.encode(chunks["text"].tolist(), batch_size=128, show_progress_bar=False, normalize_embeddings=True)
    out = pd.DataFrame(vecs, columns=[f"e{i}" for i in range(vecs.shape[1])])
    out.attrs["key"] = str(key)
    out.to_parquet(CHUNK_EMBED_CACHE)
    return vecs


chunk_emb = embed_chunks()
# Embedding dell'articolo = media dei suoi blocchi (raggruppati per doc_idx), poi rinormalizzata a lunghezza 1
emb_full = pd.DataFrame(chunk_emb).groupby(chunks["doc_idx"].values).mean().to_numpy()
emb_full = emb_full / np.linalg.norm(emb_full, axis=1, keepdims=True)
# Versione breve: un embedding per articolo, il testo sta tutto nella finestra del modello
emb_short = embedding_model.encode(df["text_short"].tolist(), batch_size=128, show_progress_bar=False, normalize_embeddings=True)
print(f"breve: {emb_short.shape} · completo: {emb_full.shape} (da {len(chunk_emb)} blocchi)")

## 4. BERTopic: stessi parametri del NYT, su più seed

**Come funziona BERTopic.** Il modello raggruppa gli articoli in tre passaggi: UMAP riduce gli embedding da 768 a 5 dimensioni; HDBSCAN cerca nello spazio ridotto le zone dense di articoli, che diventano topic, e lascia fuori come outlier (topic −1) gli articoli che non appartengono a nessuna; c-TF-IDF estrae per ogni topic le parole che lo distinguono dagli altri.

**Parametri identici al NYT** (notebook 04, §5), così l'unica differenza tra i modelli è il testo:

- `UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric="cosine")`;
- `min_topic_size = 15`: un topic deve contenere almeno 15 articoli;
- `CountVectorizer(ngram_range=(1, 2), min_df=3)`: parole singole e coppie di parole (`los angeles`, `national guard`) presenti in almeno 3 articoli;
- outlier riassegnati con `reduce_outliers(strategy="embeddings", threshold=0.4)`: ogni outlier va al topic con il centro più simile, se la similarità supera 0,4; poi le parole dei topic vengono ricalcolate.

Le stopword sono quelle del NYT più alcune parole di servizio frequenti nel Guardian (`told`, `guardian`, `people`, `just`, `like`, `according`, `including`). `ice` è esclusa perché compare in tutti gli articoli e non distingue nessun topic. Le parole di ciascun modello sono calcolate sul testo della sua versione: il testo completo ha un vocabolario circa dieci volte più ampio.

**Più seed.** UMAP è stocastico: con un seed diverso la proiezione a 5 dimensioni cambia, e HDBSCAN può trovare gruppi diversi. Un confronto su un solo seed può dipendere dal caso: con il seed 42 il modello breve concentra il 97,6% degli articoli in un solo topic, con altri seed no. Ogni versione è quindi addestrata con 5 seed (`SEEDS = [0, 1, 7, 42, 123]`), 10 modelli in tutto. Il modello di riferimento per i §6-7 è quello sul testo completo con `REFERENCE_SEED = 42`, lo stesso seed del NYT.

**Collasso totale.** Se un modello trova al più un topic oltre agli outlier, il c-TF-IDF di BERTopic ha meno di 3 "documenti" (uno per gruppo) e `min_df=3` lo fa fallire. Succede con mpnet sul testo breve (seed 7): `fit_topics` in quel caso riaddestra lo stesso modello con `min_df=1`. La partizione non cambia, perché il vectorizer serve solo a scegliere le parole e UMAP ha il seed fissato; cambia solo il vocabolario delle parole del topic. Il modello collassato resta nel confronto del §5 invece di interrompere il notebook.

In [ ]:
# Stopword inglesi + quelle del NYT + parole di servizio del Guardian; 'ice' esclusa perché è in ogni articolo
CUSTOM_STOPWORDS = list(ENGLISH_STOP_WORDS | {"ice", "new", "said", "says", "mr", "ms", "year", "years", "old",
                                              "told", "guardian", "people", "just", "like", "according", "including"})
# Parametri identici al NYT (notebook 04): minimo 15 articoli per topic, soglia 0,4 per riassegnare gli outlier
MIN_TOPIC_SIZE = 15
OUTLIER_THRESHOLD = 0.4
# Cinque seed per versione; il modello di riferimento usa il seed del NYT
SEEDS = [0, 1, 7, 42, 123]
REFERENCE_SEED = 42


def fit_topics(docs: list[str], embeddings: np.ndarray, seed: int) -> tuple[BERTopic, list[int], np.ndarray]:
    """Addestra un modello BERTopic con i parametri del NYT e il seed dato.

    Restituisce il modello, il topic di ogni documento dopo la riassegnazione degli outlier e un
    vettore booleano con gli articoli che HDBSCAN aveva lasciato fuori (topic -1). Gli outlier
    vanno al topic più simile se la similarità supera OUTLIER_THRESHOLD; dopo la riassegnazione
    le parole dei topic vengono ricalcolate con lo stesso vectorizer.
    """
    def build(min_df: int) -> tuple[BERTopic, CountVectorizer]:
        vectorizer = CountVectorizer(stop_words=CUSTOM_STOPWORDS, ngram_range=(1, 2), min_df=min_df)
        model = BERTopic(
            embedding_model=embedding_model,
            umap_model=UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric="cosine", random_state=seed),
            vectorizer_model=vectorizer,
            min_topic_size=MIN_TOPIC_SIZE,
        )
        return model, vectorizer

    model, vectorizer = build(min_df=3)
    try:
        raw, _ = model.fit_transform(docs, embeddings)
    except ValueError:
        # Collasso totale: al più 2 gruppi (outlier inclusi), meno di min_df=3 "documenti" per c-TF-IDF.
        # Il vectorizer non entra nel clustering: con min_df=1 la partizione è la stessa e il collasso resta nel §5
        log.warning("seed %d: collasso totale, c-TF-IDF ricalcolato con min_df=1", seed)
        model, vectorizer = build(min_df=1)
        raw, _ = model.fit_transform(docs, embeddings)
    was_outlier = np.array(raw) == -1
    if was_outlier.any():
        topics = model.reduce_outliers(docs, raw, strategy="embeddings", embeddings=embeddings, threshold=OUTLIER_THRESHOLD)
        model.update_topics(docs, topics=topics, vectorizer_model=vectorizer)
    else:
        topics = raw
    return model, list(topics), was_outlier


# 5 seed × 2 versioni = 10 modelli; per ciascuno si tengono modello, topic, embedding e outlier originali
runs = {}
for seed in SEEDS:
    for name, col, emb in [("breve", "text_short", emb_short), ("completo", "text_full", emb_full)]:
        model, topics, was_outlier = fit_topics(df[col].tolist(), emb, seed)
        runs[(name, seed)] = {"model": model, "topics": np.array(topics), "emb": emb, "was_outlier": was_outlier}
        log.info("seed %3d · %-8s: %2d topic, outlier %.1f%% -> %.1f%%", seed, name, len(set(topics)) - (-1 in topics),
                 100 * was_outlier.mean(), 100 * np.mean(np.array(topics) == -1))

Il log riporta, per ogni seed e versione, il numero di topic e la quota di outlier prima e dopo la riassegnazione. Il testo completo dà 23-25 topic con ogni seed, con il 31-37% di outlier prima della riassegnazione e quasi nessuno dopo. Il testo breve dà 22-27 topic con i seed 0, 1, 42 e 123, con il 38-44% di outlier; con il seed 7 trova invece 2 topic e nessun outlier, perché un unico gruppo assorbe il 98,9% degli articoli. È il collasso misurato nel §5; il messaggio "collasso totale" indica che per quel modello il c-TF-IDF è stato ricalcolato con `min_df=1` (vedi sopra).

Con MiniLM il breve collassava con due seed (7 e 42), con mpnet con uno. Il collasso del seed 7 è però al limite: dipende da differenze minime negli embedding. In una prova con embedding calcolati a blocchi di 64 testi invece che di 128 (differenza massima tra i vettori 3,7·10⁻⁷) lo stesso seed non collassa e dà 23 topic. Il collasso va quindi letto come un rischio del testo breve, non come una proprietà di un seed preciso.

Gli avvisi di BERTopic sulla "custom list of topic assignments" sono attesi: segnalano che le assegnazioni sono state modificate da `reduce_outliers`, e qui non viene applicata nessuna riduzione successiva del numero di topic.

## 5. Confronto tra testo breve e testo completo

Ogni modello (versione × seed) è descritto con le stesse misure:

- **numero di topic e outlier** (prima e dopo la riassegnazione): quanto del corpus il modello riesce a organizzare;
- **topic più grande**: quota del corpus nel topic più numeroso. Si parla di **collasso** quando supera il 50%: il modello non ha trovato una struttura e ha messo quasi tutto insieme;
- **coerenza (NPMI)**: per le 10 parole più rappresentative di ogni topic, quanto spesso compaiono insieme rispetto a quanto ci si aspetterebbe per caso. La co-occorrenza si conta nei blocchi da circa 150 parole del §2, un riferimento uguale per le due versioni e indipendente dal testo su cui sono addestrate. Va da −1 a 1: più è alta, più le parole del topic stanno davvero insieme;
- **diversità**: quota di parole distinte tra le top-10 di tutti i topic. Bassa = topic che ripetono le stesse parole;
- **silhouette** (distanza coseno) negli embedding completi, per entrambe le versioni: quanto i gruppi sono separati rispetto al contenuto dell'intero articolo. Calcolata sugli embedding completi, favorisce per costruzione il modello completo;
- **stabilità**: AMI (*adjusted mutual information*) tra le partizioni della stessa versione ottenute con seed diversi. 1 = stessi gruppi qualunque sia il seed, 0 = accordo pari al caso. Un modello i cui topic cambiano a ogni esecuzione non è interpretabile, per quanto coerenti siano le parole.

Il §5.1 scompone la stabilità coppia per coppia di seed, per distinguere un modello instabile da uno che a volte collassa.

In [ ]:
from sklearn.metrics import silhouette_score

# Parole per topic usate per NPMI e diversità
TOP_WORDS = 10
# Riferimento per la NPMI: presenza (binary) di ogni parola in ogni blocco, comune alle due versioni
ref_vectorizer = CountVectorizer(stop_words=CUSTOM_STOPWORDS, ngram_range=(1, 2), min_df=3, binary=True)
ref_matrix = ref_vectorizer.fit_transform(chunks["text"]).tocsc()
ref_vocab = ref_vectorizer.vocabulary_
n_ref = ref_matrix.shape[0]


def topic_words(model: BERTopic, topic: int, n: int = TOP_WORDS) -> list[str]:
    """Le prime n parole di un topic secondo c-TF-IDF (esclusi gli slot vuoti)."""
    return [w for w, _ in model.get_topic(topic)[:n] if w]


def npmi(words: list[str]) -> float:
    """Coerenza NPMI media sulle coppie di parole di un topic.

    Per ogni coppia (a, b): log(P(a,b) / (P(a) P(b))) / -log P(a,b), con le probabilità stimate
    come quota di blocchi che contengono le parole. Una coppia mai insieme vale -1. Le parole
    assenti dal vocabolario di riferimento vengono ignorate.
    """
    cols = [ref_vocab[w] for w in words if w in ref_vocab]
    scores = []
    for a, b in combinations(cols, 2):
        pa, pb = ref_matrix[:, a].sum() / n_ref, ref_matrix[:, b].sum() / n_ref
        pab = ref_matrix[:, a].multiply(ref_matrix[:, b]).sum() / n_ref
        scores.append(-1.0 if pab == 0 else np.log(pab / (pa * pb)) / -np.log(pab))
    return float(np.mean(scores)) if scores else np.nan


def summarize(name: str, seed: int) -> dict:
    """Riassume un modello (versione, seed) con le misure del §5, in un dizionario.

    Topic, outlier prima e dopo la riassegnazione, dimensione del topic più grande e mediana,
    NPMI media, diversità delle parole e silhouette sugli embedding completi (senza outlier).
    """
    run = runs[(name, seed)]
    model, topics = run["model"], run["topics"]
    ids = sorted(t for t in set(topics) if t != -1)
    words = [topic_words(model, t) for t in ids]
    sizes = pd.Series(topics[topics != -1]).value_counts()
    assigned = topics != -1
    return {
        "versione": name, "seed": seed,
        "topic": len(ids),
        "outlier prima %": 100 * run["was_outlier"].mean(),
        "outlier dopo %": 100 * (~assigned).mean(),
        "topic più grande %": 100 * sizes.max() / len(topics),
        "dimensione mediana": sizes.median(),
        "coerenza NPMI": np.nanmean([npmi(w) for w in words]),
        "diversità": len({w for ws in words for w in ws}) / sum(len(ws) for ws in words),
        "silhouette (emb. completi)": silhouette_score(emb_full[assigned], topics[assigned], metric="cosine"),
    }


comparison = pd.DataFrame([summarize(name, seed) for name in ("breve", "completo") for seed in SEEDS]).set_index(["versione", "seed"])
display(comparison.round(3))

metric_cols = ["topic", "outlier prima %", "topic più grande %", "coerenza NPMI", "diversità", "silhouette (emb. completi)"]
summary = comparison.groupby("versione")[metric_cols].agg(["mean", "min", "max"]).round(3)
# Stabilità: AMI medio su tutte le coppie di seed della stessa versione
stability = {
    name: np.mean([adjusted_mutual_info_score(runs[(name, a)]["topics"], runs[(name, b)]["topics"]) for a, b in combinations(SEEDS, 2)])
    for name in ("breve", "completo")
}
# Collasso: un solo topic con più di metà del corpus
collapses = (comparison["topic più grande %"] > 50).groupby("versione").sum()
# Accordo tra le due versioni a parità di seed
between = np.mean([adjusted_mutual_info_score(runs[("breve", s)]["topics"], runs[("completo", s)]["topics"]) for s in SEEDS])
display(summary.T)
print(f"Stabilità (AMI medio tra seed): breve {stability['breve']:.3f} · completo {stability['completo']:.3f}")
print(f"Collassi (un topic > 50% del corpus): breve {collapses['breve']}/{len(SEEDS)} · completo {collapses['completo']}/{len(SEEDS)}")
print(f"AMI medio tra breve e completo a parità di seed: {between:.3f}")

Prima tabella: le misure per ciascuno dei 10 modelli. Seconda tabella: media, minimo e massimo per versione. Le righe stampate: stabilità media, collassi e accordo tra le due versioni.

- Il testo completo è regolare: 23-25 topic, topic più grande al 16-20%, nessun collasso, NPMI 0,22-0,23, diversità 0,66-0,70, silhouette 0,08.
- Il testo breve, tolto il seed 7, trova 22-27 topic con NPMI appena più alta del completo (0,23) e parole un po' più varie (diversità 0,69-0,73). Con il seed 7 collassa: 2 topic, il più grande con il 98,9% degli articoli. La NPMI (0,30) e la diversità (0,95) di quel modello sono alte solo perché misurate su due liste di parole, una delle quali descrive un gruppo di 19 articoli: non indicano topic migliori.
- Stabilità media (0,487 contro 0,842) e collassi (1 su 5 contro 0) restano la differenza più netta; il §5.1 mostra da dove viene il valore del breve.
- La silhouette è bassa per entrambi (0,02-0,04 contro 0,08): gli articoli sono tutti su ICE e i confini tra topic sono sfumati.
- L'AMI tra breve e completo a parità di seed (0,379) indica che le due versioni raggruppano gli articoli in modo in buona parte diverso: non trovano gli stessi topic con parole diverse, ma partizioni diverse.

### 5.1 Stabilità coppia per coppia

L'AMI medio della tabella precedente è la media su tutte le 10 coppie di seed. Se una versione a volte collassa, le coppie tra un modello collassato e uno riuscito hanno AMI vicino a zero e abbassano la media anche se i modelli riusciti sono stabili. Per distinguere i due casi la cella mostra la matrice dell'AMI per ogni coppia di seed e calcola, per entrambe le versioni, l'AMI medio sui soli seed in cui il breve non collassa.


In [ ]:
# Matrice AMI per ogni coppia di seed, per ciascuna versione (diagonale = 1 per definizione)
ami_matrix = {
    name: pd.DataFrame(
        [[1.0 if a == b else adjusted_mutual_info_score(runs[(name, a)]["topics"], runs[(name, b)]["topics"]) for b in SEEDS]
         for a in SEEDS],
        index=SEEDS, columns=SEEDS,
    )
    for name in ("breve", "completo")
}
# Seed in cui il breve collassa (un topic > 50%) e seed in cui non collassa
collapsed = comparison.loc["breve"].index[comparison.loc["breve", "topic più grande %"] > 50].tolist()
stable_seeds = [s for s in SEEDS if s not in collapsed]
for name in ("breve", "completo"):
    pairs = [ami_matrix[name].loc[a, b] for a, b in combinations(stable_seeds, 2)]
    print(f"{name}: AMI medio sui seed {stable_seeds} = {np.mean(pairs):.3f}")

fig, axes = plt.subplots(1, 2, figsize=(10, 4.4))
for ax, name, title in zip(axes, ("breve", "completo"), ("Short text", "Full text")):
    sns.heatmap(ami_matrix[name], annot=True, fmt=".2f", vmin=0, vmax=1, cmap="Blues", cbar=False, square=True, ax=ax)
    ax.set_title(title)
    ax.set_xlabel("seed")
    ax.set_ylabel("seed")
fig.suptitle("Agreement between topic partitions obtained with different seeds (AMI)", fontsize=11)
plt.tight_layout()
fig.savefig(FIG_DIR / "guardian_topics_seed_ami.png", dpi=200, bbox_inches="tight")
plt.show()

Le due matrici riportano l'AMI per ogni coppia di seed. Il testo completo ha valori tra 0,82 e 0,87 in tutte le coppie. Nel testo breve il seed 7, collassato, ha AMI 0,03-0,04 con tutti gli altri; tra i quattro seed non collassati l'accordo va da 0,76 a 0,88. Sui seed non collassati l'AMI medio è 0,787 per il breve e 0,853 per il completo.

Il testo breve quindi non è instabile in generale: è esposto al collasso, e anche quando non collassa le sue partizioni sono un po' meno stabili di quelle del completo (con MiniLM le due versioni erano alla pari, 0,835 contro 0,839). Il testo completo non collassa mai. Il dato è coerente con il NYT: il BERTopic su testo breve del notebook 04 è stabile su 10 seed e non collassa, quindi il testo breve non porta necessariamente al collasso, ma su questo corpus lo ha fatto in 1 caso su 5. La figura è salvata in `docs/figures/guardian_topics_seed_ami.png`.

## 6. I topic del modello sul testo completo

Modello di riferimento: testo completo, `REFERENCE_SEED = 42`, fissato prima di vedere i risultati perché è il seed usato per il NYT. Non è il "migliore" dei cinque: con il testo completo le partizioni dei diversi seed concordano (AMI 0,82-0,87, §5.1), quindi la scelta del seed cambia poco.

La tabella riporta per ogni topic, in ordine di dimensione: numero di articoli, quota di live-blog, le 8 parole più rappresentative (c-TF-IDF) e i titoli dei 3 articoli più vicini al centro del topic. I titoli sono più leggibili delle parole e servono a dare un nome al topic. Il centro è la media degli embedding degli articoli del topic; i più vicini sono quelli con la similarità del coseno più alta.

In [ ]:
def describe_topics(name: str, seed: int) -> pd.DataFrame:
    """Tabella descrittiva dei topic di un modello (versione, seed), in ordine di dimensione.

    Per ogni topic: articoli, quota di live-blog, prime 8 parole c-TF-IDF e titoli dei 3 articoli
    più vicini al centro del topic (media degli embedding, similarità del coseno).
    """
    run = runs[(name, seed)]
    model, topics, emb = run["model"], run["topics"], run["emb"]
    rows = []
    for t in sorted(set(topics) - {-1}, key=lambda t: -(topics == t).sum()):
        idx = np.where(topics == t)[0]
        centroid = emb[idx].mean(axis=0)
        closest = idx[np.argsort(-(emb[idx] @ centroid))[:3]]
        rows.append({
            "topic": t, "articoli": len(idx), "% live": round(100 * df["is_live"].values[idx].mean()),
            "parole": ", ".join(topic_words(model, t, 8)),
            "titoli più centrali": " | ".join(df["headline"].values[closest]),
        })
    return pd.DataFrame(rows).set_index("topic")


# Modello di riferimento: testo completo, seed 42
final = runs[("completo", REFERENCE_SEED)]
topic_table = describe_topics("completo", REFERENCE_SEED)
df["topic"] = final["topics"]
# Articoli che HDBSCAN aveva lasciato fuori e che reduce_outliers ha assegnato al topic più simile: appartenenza più debole
df["topic_was_outlier"] = final["was_outlier"]
# Etichetta leggibile di ogni topic: numero + prime 3 parole
labels = {t: f"{t}: " + ", ".join(topic_words(final["model"], t, 3)) for t in topic_table.index}
labels[-1] = "-1: outlier"
df["topic_label"] = df["topic"].map(labels)
display(topic_table)

23 topic, da 336 a 19 articoli, di due tipi.

- Temi trasversali, presenti in più periodi: casi individuali di famiglie e deportazioni (0, 336 articoli), condizioni di detenzione (1), politica di arresti dell'amministrazione e del DHS (5), città santuario (8), deportazioni verso paesi terzi e asilo (10), separazione delle famiglie (6), detenzione di donne e bambini, con Dilley (14), sorveglianza e dati con Palantir (11), decessi in custodia (16).
- Eventi del 2025-2026: Minneapolis (2, e 18 sulla risposta delle comunità), sparatorie di agenti (4), Khalil e le università (3), proteste "No Kings" (9), Ábrego García (12), Los Angeles e Guardia nazionale (15), processi ai manifestanti (13), Delaney Hall (17), Liam Ramos (19), scuole (20), "Alligator Alcatraz" (21), finanziamento del DHS (22).

Il topic 0 è più grande che con MiniLM (336 articoli contro 202): raccoglie storie individuali di tutti i periodi. Il topic 7 (`trump, president, house`) è composto per il 31% da live-blog: è un contenitore di cronaca politica generale più che un tema su ICE, e raccoglie 25 dei 39 live-blog; gli altri topic ne hanno al più l'8%.

Rispetto alla prima esecuzione con MiniLM (25 topic) l'accordo tra i due modelli di riferimento è AMI 0,60, 0,87 sugli articoli che nessuno dei due aveva lasciato tra gli outlier: i temi sono in gran parte gli stessi, i confini tra topic vicini cambiano. Scompaiono come topic propri i raid sui luoghi di lavoro e i tribunali dell'immigrazione; compaiono i processi ai manifestanti, le scuole e il finanziamento del DHS.

Per confronto, i topic del modello sul testo breve, sullo stesso corpus. Si mostra il seed con più topic (123, 27 topic), cioè il caso più favorevole al testo breve tra quelli provati.

In [ ]:
# Seed del breve con più topic: il caso più favorevole al testo breve
best_short_seed = comparison.loc["breve", "topic"].idxmax()
print(f"seed {best_short_seed}")
display(describe_topics("breve", best_short_seed))

Il testo breve trova soprattutto eventi e persone: Khalil (0), Minneapolis (8), l'uccisione di Renee Good (11), le sparatorie (9), Ábrego García (18), Liam Ramos (23), Tom Homan e la direzione dell'agenzia (25), Delaney Hall (10), Los Angeles (13, 19). Ha anche temi trasversali (detenzione 1, decessi in custodia 6, separazione delle famiglie 4, sorveglianza 5, deportazioni verso paesi terzi 16), più che con MiniLM. Il topic 12 (`trump, happened, live`) è fatto di riepiloghi e live-blog ("Trump news at a glance"), cioè di testi che nella versione breve si riconoscono solo dal formato. La differenza resta coerente con il testo usato: titolo e attacco nominano l'evento, il resto dell'articolo sviluppa il tema.

### 6.1 Peso dei topic per periodo

I periodi hanno dimensioni molto diverse (il 2025-presente da solo è il 72% del corpus), quindi ogni colonna della heatmap è normalizzata sul proprio periodo: il valore è la quota degli articoli di quel periodo assegnata a ciascun topic, e ogni colonna somma a 100. I topic sono ordinati per la quota massima raggiunta in un periodo. Sotto ogni periodo è indicato il numero di articoli: nei periodi più piccoli un punto percentuale corrisponde a uno o due articoli.

In [ ]:
# Quota di ogni topic sugli articoli del periodo (colonne normalizzate a 100)
topic_period = pd.crosstab(df["topic_label"], df["period"], normalize="columns").reindex(columns=PERIOD_ORDER) * 100
topic_period = topic_period.drop(index=labels[-1], errors="ignore")
# Ordine: topic con la quota massima più alta in alto
topic_period = topic_period.loc[topic_period.max(axis=1).sort_values(ascending=False).index]
period_sizes = df["period"].value_counts().reindex(PERIOD_ORDER)

fig, ax = plt.subplots(figsize=(10, 0.42 * len(topic_period) + 2))
sns.heatmap(topic_period, annot=True, fmt=".1f", cmap="Blues", cbar_kws={"label": "% of the period's articles"}, ax=ax)
ax.set_xticklabels([f"{p}\n(n={period_sizes[p]})" for p in PERIOD_ORDER], rotation=0)
ax.set_title("Guardian (full text): share of each topic by period (%)")
ax.set_xlabel("")
ax.set_ylabel("")
plt.tight_layout()
fig.savefig(PLOTS_DIR / "topics_by_period.png", dpi=150)
fig.savefig(FIG_DIR / "guardian_topics_by_period.png", dpi=200, bbox_inches="tight")
plt.show()

Lo spostamento tematico si legge per colonne. Prima del 2017 dominano le storie individuali (topic 0: 35,2% nel 2003-2016), la detenzione (1: 15,7%) e la detenzione di donne e bambini (14: 14,8%); nel 2017-2020 restano le storie individuali (31,4%) e crescono città santuario (8: 12,4%) e separazione delle famiglie (6: 10,2%). Nel 2021-2024 pesano detenzione (20,0%), storie individuali (18,9%), asilo e deportazioni verso paesi terzi (10: 16,7%), città santuario (16,7%) e sorveglianza e dati (11: 12,2%), ma su 90 articoli: il 12,2% sono 11 articoli. Nel 2025-presente il peso si distribuisce su molti topic più piccoli, legati a eventi e luoghi interni agli USA (Minneapolis 8,8%, sparatorie 7,5%, Khalil 6,2%), e solo le storie individuali superano il 10% (15,3%). I topic-evento del 2025-2026 valgono 0 o quasi nei periodi precedenti per costruzione. Lo spostamento dalla famiglia e dal confine agli eventi nelle città statunitensi è lo stesso della prima esecuzione con MiniLM ed è coerente con la NER sul NYT (notebook 06). La figura è salvata anche in `docs/figures/guardian_topics_by_period.png`.

## 7. Salvataggio

Si salvano i risultati del modello di riferimento (testo completo, `REFERENCE_SEED = 42`), che i notebook successivi usano come descrizione tematica del corpus Guardian.

- `ice_topics.parquet`: un articolo per riga, con `id`, data, anno, periodo, `is_live`, `topic`, `topic_label` (numero + prime 3 parole) e `topic_was_outlier`. Quest'ultima colonna vale `True` per il 33% degli articoli, quelli che HDBSCAN aveva lasciato fuori e che `reduce_outliers` ha assegnato al topic più simile: la loro appartenenza è più debole. Per analisi che richiedono appartenenze solide si usano solo le righe con `topic_was_outlier == False`.
- `ice_topic_info.csv`: la tabella dei topic del §6 (articoli, quota di live-blog, parole, titoli più centrali).

I modelli degli altri seed e quelli sul testo breve non vengono salvati: servono solo al confronto del §5, che si riproduce rieseguendo il notebook (i risultati sono deterministici a parità di seed).

In [ ]:
# Una riga per articolo con il topic del modello di riferimento e il flag di riassegnazione
out = df[["id", "pub_date", "year", "period", "is_live", "topic", "topic_label", "topic_was_outlier"]].copy()
out.to_parquet(OUT_TOPICS, index=False)
# Tabella descrittiva dei topic (§6), per consultazione e per i notebook successivi
topic_table.reset_index().to_csv(OUT_TOPIC_INFO, index=False)
log.info("Scritti %s (%d articoli, %d topic) e %s", OUT_TOPICS.relative_to(BASE_DIR), len(out), len(topic_table), OUT_TOPIC_INFO.relative_to(BASE_DIR))

## 8. Lettura dei risultati

Run del 2026-09-30 con MiniLM, rieseguito il 2026-10-08 con risultati identici. Dal 2026-10-13 il notebook usa `all-mpnet-base-v2`, come i notebook 04 e 09; i numeri sotto sono quelli di mpnet, tra parentesi quelli di MiniLM dove la conclusione cambia di grado.

**Il testo completo è più robusto, non più coerente.**

- **Robustezza.** È la differenza netta, con entrambi i modelli di embedding. Con il testo completo i topic sono quasi gli stessi a ogni seed (23-25 topic, AMI tra seed 0,84, nessun collasso). Con il testo breve l'AMI medio è 0,49 (MiniLM 0,39) e il valore viene soprattutto dai collassi: con il seed 7 il modello mette il 98,9% degli articoli in un solo topic (con MiniLM i seed collassati erano due). Sui seed riusciti le partizioni del breve concordano un po' meno di quelle del completo (AMI 0,79 contro 0,85; con MiniLM erano alla pari). Il collasso è al limite: lo stesso seed non collassa se gli embedding cambiano di 10⁻⁷ (§4). Il testo breve non è instabile in generale: è esposto al collasso.
- **Coerenza delle parole.** Nessun vantaggio del completo. Quando il modello breve non collassa le sue parole sono appena più coerenti (NPMI 0,23 contro 0,22-0,23) e un po' più varie tra topic (diversità 0,69-0,73 contro 0,66-0,70). Il testo completo usa un vocabolario generico (`immigration`, `trump`, `detention`) che ricorre in molti topic. Con mpnet la distanza tra le due versioni è più piccola che con MiniLM.
- **Separazione dei gruppi.** Silhouette bassa per entrambi (0,08 il completo, 0,02-0,04 il breve): gli articoli sono tutti su ICE e i confini tra topic sono sfumati. La misura è calcolata sugli embedding completi e favorisce per costruzione il modello completo.
- **Contenuti.** Il breve organizza la copertura soprattutto per vicende (Khalil, Minneapolis, Renee Good, Ábrego García), il completo trova le stesse vicende più temi trasversali: condizioni di detenzione, decessi in custodia, sorveglianza e dati (Palantir), città santuario, deportazioni verso paesi terzi. Con mpnet anche il breve ha più temi trasversali che con MiniLM. Le due versioni producono partizioni in buona parte diverse (AMI tra loro 0,38).
- **Cambio di modello.** I topic del testo completo con mpnet e con MiniLM concordano con AMI 0,60 (0,87 sugli articoli non outlier in entrambi): stessi temi principali, confini diversi tra topic vicini, e la stessa lettura per periodo (§6.1).

**Cosa significa per il NYT.** I topic del NYT (notebook 04) sono calcolati su testo breve. Il confronto non li rende inaffidabili: il modello NYT è stabile su 10 seed e non collassa (notebook 04), che è la verifica indicata dal §5.1. Suggerisce però che i topic del NYT descrivano soprattutto vicende, e che temi che emergono solo nel corpo degli articoli possano esservi sottorappresentati. Il testo NYT del 2025-2026 conta in mediana 34 parole, meno della versione breve usata qui (91): l'orientamento verso le vicende potrebbe essere più marcato proprio nel periodo più recente. È un'ipotesi da verificare, non un risultato.

**Limiti.**

- Il 33% degli articoli era outlier ed è stato assegnato al topic più simile (`topic_was_outlier`). Con gli embedding mediati sui blocchi le similarità sono tutte alte (coseno medio tra articoli 0,54 contro 0,40 del breve) e la soglia 0,4, presa dal NYT, non filtra quasi nulla. Nei topic 18, 8 e 5 più della metà degli articoli è riassegnata.
- Il topic 7 (`trump, president, house`) è composto per il 31% da live-blog: è un contenitore di cronaca politica generale, non un tema su ICE.
- I periodi anteriori al 2025 sono piccoli (108 / 283 / 90 articoli): le quote per periodo della heatmap sono incerte, e i topic-evento del 2025-2026 valgono 0 nei periodi precedenti per costruzione.
- La media dei blocchi appiattisce gli articoli che trattano più temi, come i live-blog; aggregazioni alternative (media pesata, più vettori per articolo) non sono state provate.
- La versione breve (91 parole) è più lunga del testo NYT (39 parole, 34 dal 2025): il confronto misura l'effetto della lunghezza sul Guardian e si trasferisce al NYT solo come indicazione di tendenza.
- I blocchi da 150 parole sono stati scelti per la finestra di MiniLM (256 token) e mantenuti con mpnet (384 token) per cambiare un solo elemento; blocchi più lunghi non sono stati provati.
- Cinque seed per versione: la frequenza del collasso del breve (1 su 5, 2 su 5 con MiniLM) è una stima grossolana, e il collasso dipende anche da differenze numeriche minime negli embedding (§4).
- Il corpus dipende dal filtro di rilevanza del notebook 05 (precisione stimata 0,90, più bassa nel 2021-2024).